In [15]:
import os
import re
import pandas as pd
from bs4 import BeautifulSoup
from datetime import date, datetime

DATA1_DIR = "Data/Cau1"
OUT1_DIR = "Cau1"
os.makedirs(OUT1_DIR, exist_ok=True)

### Cau 1:
a

In [26]:
def boc_tach_1_file(duong_dan):
    with open(duong_dan, encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html5lib")

    rows = []
    for card in soup.select(".job-card"):
        ma_tin = card.get("data-job")
        vi_tri = card.find(class_="job-title").get_text(strip=True)

        cong_ty = card.find(class_="company").get_text(strip=True)
        muc_luong_raw = card.find(class_="pay").get_text(strip=True)
        muc_luong = re.sub(r"^Lương:\s*","",muc_luong_raw).strip()
        han_nop_raw = card.find(class_="deadline").get_text(strip=True)
        han_nop = re.sub(r"^Hạn nộp:\s*","",muc_luong_raw).strip()
        perks_ul=card.find(class_="perks")
        if perks_ul is not None:
            items = [li.get_text(strip=True) for li in perks_ul.find_all("li")]
        else:
            items=[]
        phuc_loi=" | ".join(items)
        

        rows.append({
            "ma_tin": ma_tin,
            "vi_tri": vi_tri,
            "cong_ty": cong_ty,
            "muc_luong": muc_luong,
            "han_nop": han_nop,
            "phuc_loi": phuc_loi,
        })
    return rows

all_rows=[]
danh_sach_goc = boc_tach_1_file(r"Data/Cau1/viec_lam_p1.htm") + boc_tach_1_file(r"Data/Cau1/viec_lam_p2.htm")

print("Số dòng dữ liệu:", len(danh_sach_goc))  # phải đúng bằng 8
assert len(danh_sach_goc) == 8, "Đề bài yêu cầu đúng 8 dòng dữ liệu!"

# Ghi ra CSV bằng csv.DictWriter -- tự động đặt dấu ngoặc kép quanh chuỗi có dấu phẩy
with open(r"Cau1/tin_tuyen_dung_goc.csv", "w", encoding="utf-8", newline="") as f:
    ghi = csv.DictWriter(f, fieldnames=["ma_tin", "vi_tri", "cong_ty", "muc_luong", "han_nop", "phuc_loi"])
    ghi.writeheader()
    ghi.writerows(danh_sach_goc)

pd.read_csv(r"Cau1/tin_tuyen_dung_goc.csv")

Số dòng dữ liệu: 8


,ma_tin,vi_tri,cong_ty,muc_luong,han_nop,phuc_loi
0,VL01,Kỹ sư Dữ liệu Junior,Công ty Alpha Data,12 - 18 triệu,Lương: 12 - 18 triệu,Bảo hiểm | Thưởng quý
1,VL02,Chuyên viên Phân tích Web,Tập đoàn Beta Media,"8,5 - 12 triệu","Lương: 8,5 - 12 triệu",Làm việc từ xa
2,VL03,Thực tập sinh NLP,Viện Gamma AI,5 triệu,Lương: 5 triệu,NaN
3,VL04,Quản trị Cơ sở dữ liệu,Công ty Delta Soft,Thỏa thuận,Lương: Thỏa thuận,Bảo hiểm | Du lịch hằng năm | Laptop công ty
4,VL05,Lập trình viên Python,Epsilon Tech,15 - 25 triệu,Lương: 15 - 25 triệu,Thưởng dự án
5,VL06,Kỹ sư Crawling,Zeta Labs,"10,5 triệu","Lương: 10,5 triệu",Bảo hiểm | Đào tạo nội bộ
6,VL07,Chuyên viên Trực quan hóa,Eta Analytics,Thỏa thuận,Lương: Thỏa thuận,NaN
7,VL08,Trưởng nhóm Khai phá dữ liệu,Theta Corp,30 - 45 triệu,Lương: 30 - 45 triệu,Bảo hiểm | Cổ phần | Thưởng năm


#b


## Câu 2. 

In [4]:
# Đọc dữ liệu gốc
df = pd.read_csv(r"Data/Cau2/y_kien_ktx.csv")

# Đọc bảng tra emoji/emoticon -> chữ (vd: '😍' -> 'yeu_thich')
emoji_map = dict(pd.read_csv(r"Data/Cau2/bieu_tuong.csv").values)

# Đọc từ điển chuẩn hóa -> từ chuẩn (vd: 'ks' -> 'khach san')
tu_dien = dict(pd.read_csv(r"Data/Cau2/tu_dien_ktx.csv").values)

# Đọc danh sách stop word (mỗi dòng 1 từ)
with open(r"Data/Cau2/tu_dung.txt", encoding="utf-8") as f:
    stop_words = set(dong.strip() for dong in f if dong.strip())

print("Số dòng review:", len(df))
print("Bảng emoji:", emoji_map)
print("Từ điển chuẩn hóa:", tu_dien)
print("Số lượng stop word:", len(stop_words))
df

Số dòng review: 12
Bảng emoji: {'😊': 'hài_lòng', '👍': 'tốt', '😡': 'bực_xúc', ':)': 'vui', ':(': 'thất_vọng', ':))': 'rất_vui', ':((': 'rất_thất_vọng'}
Từ điển chuẩn hóa: {'wc': 'nhà_vệ_sinh', 'ktx': 'ký_túc_xá', 'wc chung': 'nhà_vệ_sinh_chung', 'gui xe': 'gửi_xe', 'wifi': 'wi_fi', 'bv': 'bảo_vệ', 'ko': 'không'}
Số lượng stop word: 16


,ma_phan_hoi,noi_dung_goc
0,KT01,"<p>KTX sạch sẽ, phòng thoáng mát, rất ổn 😊. Xe..."
1,KT02,"Đăng ký phòng online nhanh, gui-xe dễ dàng :))"
2,KT03,"Wc chung tầng 2 hơi bẩn, nước yếu, nước lạnh v..."
3,KT04,Wifi mạnh nhưng bv kiểm tra thẻ ra vào hơi chậ...
4,KT05,<style>.b{color:red}</style>Phòng 305 rộng rãi...
5,KT06,Bảo vệ nhiệt tình 👍 nhưng wc hỏng 3 ngày chưa ...
6,KT07,"Khu gui xe rộng, có mái che, wifi ổn."
7,KT08,"Xem bản đồ tại www.ktx.example.vn/ban-do, phòn..."
8,KT09,KTX có nhà ăn đa dạng món; wc chung sạch sẽ 😊
9,KT10,"Nước nóng không ổn định, ko có ổ cắm gần giườn..."


### a) 

In [6]:
DATA2_DIR="Data/Cau2"
OUT2_DIR="Cau2"
os.makedirs(OUT2_DIR, exist_ok=True)
def load_map(csv_path):
    m=pd.read_csv(csv_path, encoding="utf-8")
    return dict(zip(m.iloc[:,0].astype(str),m.iloc[:,1].astype(str)))

In [9]:
def buoc1_lam_sach_so_bo(text):
    # 1) Xóa thẻ HTML dạng <...> (vd <p>, </div>) bằng regex không tham lam '[^>]+'
    text = re.sub(r"<[^>]+>", " ", text)

    # 2) Chuyển về chữ thường
    text = text.lower()

    # 3) Xóa URL dạng http(s)://... hoặc www....  \S+ = chuỗi không có khoảng trắng
    text = re.sub(r"(https?://\S+|www\.\S+)", " ", text)

    # 4) Xóa chữ số (ví dụ: '30', '100', '1', '2')
    text = re.sub(r"\d+", " ", text)

    # 5) Chuẩn hóa khoảng trắng: nhiều space/tab liên tiếp -> 1 space, cắt 2 đầu
    text = re.sub(r"\s+", " ", text).strip()

    return text
def count_so_cau(text):
    parts=re.split(r"[.!?]+",text)
    return sum(1 for p in parts if any(ch.isalpha() for ch in p))

df_raw= pd.read_csv(os.path.join(DATA2_DIR, "lam_sach_1.csv"), encoding="utf-8")
df_a=pd.DataFrame()
df_a["ma_phan_hoi"]=df_raw["ma_phan_hoi"]
df_a["van_ban_1"]=df_raw["noi_dung_goc"].apply(buoc1_lam_sach_so_bo)
df_a["so_cau"]=df_a["van_ban_1"].apply(count_so_cau)
df_a.to_csv(os.path.join(OUT2_DIR,"lam_sach_1.csv"), index=False, encoding="utf-8")
df_a



,ma_phan_hoi,van_ban_1,so_cau
0,KT01,"ktx sạch sẽ, phòng thoáng mát, rất ổn 😊. xem n...",2
1,KT02,"đăng ký phòng online nhanh, gui-xe dễ dàng :))",1
2,KT03,"wc chung tầng hơi bẩn, nước yếu, nước lạnh vào...",1
3,KT04,wifi mạnh nhưng bv kiểm tra thẻ ra vào hơi chậ...,2
4,KT05,".b{color:red} phòng rộng rãi, giá hợp lý track()",1
5,KT06,bảo vệ nhiệt tình 👍 nhưng wc hỏng ngày chưa sử...,1
6,KT07,"khu gui xe rộng, có mái che, wifi ổn.",1
7,KT08,xem bản đồ tại phòng sinh hoạt chung sạch sẽ.,1
8,KT09,ktx có nhà ăn đa dạng món; wc chung sạch sẽ 😊,1
9,KT10,"nước nóng không ổn định, ko có ổ cắm gần giườn...",1


### b) 

In [25]:
emoji_map=load_map(os.path.join(DATA2_DIR, "bieu_tuong.csv"))
abbr_map=load_map(os.path.join(DATA2_DIR, "tu_dien_ktx.csv"))



def apply_emoji(text, mapping):
    # key dai hon thay truoc (":((" truoc ":(" ) de tranh vo cum
    for k in sorted(mapping.keys(), key=len, reverse=True):
        text = text.replace(k, " " + mapping[k] + " ")
        return text

def remove_punct_keep_underscore(text):
    punct = string.punctuation.replace("_",Len(punct))
    text = text.translate(str.maketrans(punct,)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def apply_dict(text, mapping):
# cum dai hon (nhieu ky tu hon) duoc thay truoc, vd "wc chung" truoc "wc"
    for k in sorted(mapping.keys(), key=len, reverse=True):
    pattern = r"\b" + re.escape(k) + r"\b"
    text = re.sub(pattern, mapping[k], text)
    return re.sub(r"\s+", " ", text).strip()

def buoc_b_clean(text):
    text = apply_emoji(text, emoji_map)
    text = remove_punct_keep_underscore(text)
    text = apply_dict(text, abbr_map)
    return text

df_a = pd.read_csv(os.path.join(OUT2_DIR, "lam_sach_1.csv"), encoding="utf-8", keep_default_na=False)

df_b = pd.DataFrame()
df_b["ma_phan_hoi"] =df_a["ma_phan_hoi"]
df_b["van_ban_2"] = df_a["van_ban_1"].apply(buoc_b_clean)
df_b.to_csv(os.path.join(OUT2_DIR, "lam_sach_2.csv"), index=False, encoding="utf-8")
df_b


SyntaxError: '(' was never closed (1267679428.py, line 14)